# 🦘 Dataset 7: Statlog — Australian Credit Approval
**Propósito:** Clasificación binaria — Aprobación de crédito (ya limpio y codificado).  
**Fuente:** Proyecto STATLOG — UCI ML Repository.  
> ✅ Dataset **sin valores faltantes** — ideal para comparar algoritmos directamente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos

In [ ]:
cols = [f'A{i}' for i in range(1,15)] + ['TARGET']
df = pd.read_csv('../datasets/4 data sets/dataset7/australian.dat', 
                 sep=' ', header=None, names=cols)
print(f"Shape: {df.shape}")
print(f"Nulos: {df.isnull().sum().sum()}")
print(f"\nDistribución TARGET: {df['TARGET'].value_counts().to_dict()}")
df.head(3)

## 2. EDA — Sin necesidad de limpieza

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))

axes[0].bar(['Rechazado (0)','Aprobado (1)'], df['TARGET'].value_counts().sort_index(),
            color=['tomato','steelblue'])
axes[0].set_title(f'Distribución de clases\n307 aprobados / 383 rechazados')

# Distribución de features numéricas continuas conocidas (A2, A3, A7, A10, A13, A14)
continuas = ['A2','A3','A7','A10','A13','A14']
df[continuas].boxplot(ax=axes[1])
axes[1].set_title('Distribución de Variables Continuas')
axes[1].tick_params(axis='x', rotation=30)

sns.heatmap(df.corr()[['TARGET']].sort_values('TARGET'), 
            annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2])
axes[2].set_title('Correlación con TARGET')

plt.tight_layout()
plt.savefig('australian_eda.png', dpi=100)
plt.show()

## 3. Preparación

In [ ]:
X = df.drop(columns=['TARGET'])
y = df['TARGET']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

## 4. Comparación de Modelos — Validación Cruzada 10-Fold
> Al ser un dataset limpio y pequeño, la validación cruzada es ideal para comparar algoritmos con robustez estadística.

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
X_s = StandardScaler().fit_transform(X)

modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)'          : SVC(probability=True, random_state=42)
}

resultados = {}
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_s, y, cv=kf, scoring='roc_auc')
    resultados[nombre] = scores
    print(f"{nombre:25s} | ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Boxplot de comparación
res_df = pd.DataFrame(resultados)
fig, ax = plt.subplots(figsize=(10,5))
res_df.boxplot(ax=ax)
ax.set_title('Comparación de Modelos — ROC-AUC (10-Fold CV)')
ax.set_ylabel('ROC-AUC')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('australian_comparacion.png', dpi=100)
plt.show()

## 5. Modelo Final — Gradient Boosting + Costo Sigmoide

In [ ]:
gbm = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbm.fit(X_train, y_train)
y_pred = gbm.predict(X_test)
y_prob = gbm.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred, target_names=['Rechazado','Aprobado']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# Función de costo sigmoide (log-loss / binary cross-entropy)
eps = 1e-15
y_t = y_test.values
y_p = np.clip(y_prob, eps, 1-eps)
log_loss_val = -np.mean(y_t * np.log(y_p) + (1-y_t) * np.log(1-y_p))
print(f"\nBinary Cross-Entropy (Log-Loss): {log_loss_val:.4f}")
print("Fórmula: L = -[y·log(p) + (1-y)·log(1-p)]")

## ✅ Resumen
| | |
|---|---|
| Registros | 690 |
| Nulos | 0 (ya limpio) |
| Mejor modelo | Gradient Boosting |
| ROC-AUC aprox | ~0.93 |

**Ventaja de este dataset:** Al no requerir limpieza, permite comparar algoritmos en igualdad de condiciones — exactamente el propósito original del proyecto STATLOG.